Partie 0 – mise en place de l’environnement 
1) Structurer le projet comme suit : 
atelier_tensorflow_iot/ 
│    
├── notebooks/ 
│   
└── atelier_tensorflow_iot.ipynb 
│ 
└── models/ 
└── modele_consommation.keras 
2) Créer le notebook atelier_tensorflow_iot.ipynb 
3) Installer et importer tensorflow,  matplotlib et numpy

In [30]:
%pip install tensorflow matplotlib numpy scikit-learn

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.2.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [31]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from sklearn.model_selection import train_test_split
import numpy as np
import matplotlib.pyplot as plt

Partie 1 – Génération du dataset 
1) Générer aléatoirement 1000 valeurs pour chacune des variables suivantes : 
a) temperature : valeurs qui suivent une loi normale avec une moyenne de 25 °C et un 
écart-type de 4 °C. 
b) humidite : valeurs réparties de façon uniforme entre 30 % et 80 %. 
c) occupants : valeurs entières choisies entre 1 et 49 inclus. 
2) Déterminer la variable consommation avec la formule suivante : 
 la consommation de base (0 °C, pas d'humidité et pièce vide) est de 50 
 chaque degré supplémentaire augmente la consommation de 5 unités 
 chaque pourcentage d'humidité en plus ajoute 1,5 unité à la consommation 
 chaque personne présente dans la pièce augmente la consommation de 4 unités 
 dans la vraie vie, une formule mathématique parfaite n'existe pas. On ajoute donc une 
petite variation aléatoire (moyenne de 0 et écart-type de 10) pour simuler des imprévus 
ou d'autres facteurs non mesurés. 
3) Rassembler les variables (temperature, humidite et occupants) dans la matrice des 
caractéristiques (features) X de taille 1000x3 en convertissant éventuellement les données au 
format (float32) optimisé pour les calculs 
4)  Créer la cible (target) y qui contiendra la variable consommation, au format float32

1) Générer aléatoirement 1000 valeurs pour chacune des variables suivantes : 

a) temperature : valeurs qui suivent une loi normale avec une moyenne de 25 °C et un 
écart-type de 4 °C. 

b) humidite : valeurs réparties de façon uniforme entre 30 % et 80 %.
 
c) occupants : valeurs entières choisies entre 1 et 49 inclus.

In [32]:
SEED = 42
rng = np.random.default_rng(SEED)
N = 1000


temperature = rng.normal(loc=25, scale=4, size=N)
humidite = rng.uniform(low=30, high=80, size=N)
occupants = rng.integers(low=1, high=50, size=N) 

2) Déterminer la variable consommation avec la formule suivante : 

 la consommation de base (0 °C, pas d'humidité et pièce vide) est de 50 
 chaque degré supplémentaire augmente la consommation de 5 unités 
 chaque pourcentage d'humidité en plus ajoute 1,5 unité à la consommation 
 chaque personne présente dans la pièce augmente la consommation de 4 unités 
 dans la vraie vie, une formule mathématique parfaite n'existe pas. On ajoute donc une 
petite variation aléatoire (moyenne de 0 et écart-type de 10) pour simuler des imprévus 
ou d'autres facteurs non mesurés. 

In [33]:
bruit = rng.normal(loc=0, scale=10, size=N)
consommation = 50 + 5 * temperature + 1.5 * humidite + 4 * occupants + bruit


3) Rassembler les variables (temperature, humidite et occupants) dans la matrice des 
caractéristiques (features) X de taille 1000x3 en convertissant éventuellement les données au 
format (float32) optimisé pour les calculs 

In [34]:
X = np.column_stack([temperature, humidite, occupants]).astype(np.float32)

4)  Créer la cible (target) y qui contiendra la variable consommation, au format float32 

In [35]:
y = consommation.astype(np.float32)

print("X shape :", X.shape, "| dtype :", X.dtype)
print("y shape :", y.shape, "| dtype :", y.dtype)
print("\nAperçu des 5 premières observations :")
for i in range(5):
    print(f"  temp={X[i,0]:5.2f}°C  hum={X[i,1]:5.2f}%  occ={int(X[i,2]):2d}  ->  conso={y[i]:7.2f}")

X shape : (1000, 3) | dtype : float32
y shape : (1000,) | dtype : float32

Aperçu des 5 premières observations :
  temp=26.22°C  hum=62.35%  occ= 5  ->  conso= 285.81
  temp=20.84°C  hum=47.12%  occ=39  ->  conso= 383.05
  temp=28.00°C  hum=50.41%  occ=41  ->  conso= 429.26
  temp=28.76°C  hum=52.00%  occ=47  ->  conso= 458.81
  temp=17.20°C  hum=36.29%  occ=16  ->  conso= 253.27


Partie 2 – Découpage Train/Test 
Diviser le dataset précédent (X et y) en deux ensembles distincts : un pour l'entraînement (train) 
et un pour le test (test). Avec les conditions suivantes : 20% des données serviront au test ; garantir 
la reproductibilité du découpage.

In [36]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

Partie 3 – Création du modèle 
1) Construire un réseau de neurones qui utilise l’architecture séquentielle suivante : 
a) couche 1 : 16 neurones avec relu comme fonction d’activation 
b) couche 2 : 8 neurones avec relu 
c) couche 3 : 1 seul neurone. 
2) Afficher un résumé textuel de l'architecture du réseau de neurones.

1) Construire un réseau de neurones qui utilise l’architecture séquentielle suivante : 
a) couche 1 : 16 neurones avec relu comme fonction d’activation 
b) couche 2 : 8 neurones avec relu 
c) couche 3 : 1 seul neurone.

In [37]:
model = keras.Sequential(
    [
        layers.Input(shape=(3,), name="features"),
        layers.Dense(16, activation="relu", name="couche_1"),
        layers.Dense(8, activation="relu", name="couche_2"),
        layers.Dense(1, name="couche_sortie"),
    ],
    name="modele_consommation"
)

model.summary()

Model: "modele_consommation"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ couche_1 (Dense)                │ (None, 16)             │            64 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ couche_2 (Dense)                │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ couche_sortie (Dense)           │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 209 (836.00 B)

 Trainable params: 209 (836.00 B)

 Non-trainable params: 0 (0.00 B)

2) Afficher un résumé textuel de l'architecture du réseau de neurones.

In [38]:
model.summary()

Model: "modele_consommation"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ couche_1 (Dense)                │ (None, 16)             │            64 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ couche_2 (Dense)                │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ couche_sortie (Dense)           │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 209 (836.00 B)

 Trainable params: 209 (836.00 B)

 Non-trainable params: 0 (0.00 B)

Partie 4 – Compilation du modèle 
Compiler le modèle avec une méthode pour ajuster les poids, une fonction permettant de mesurer 
l'erreur de prédiction et une métrique permettant de suivre l'erreur absolue moyenne. 

In [39]:
model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"],
)

print("Modèle compilé : optimiseur=adam | loss=mse | metrique=mae")

Modèle compilé : optimiseur=adam | loss=mse | metrique=mae


Partie 5 – Entraînement du modèle 
1) Entrainer le modèle avec 20 % des données d'entraînement utilisés pour la validation, 50 
passages sur les données et 16 observations traitées à la fois. Sauvegarder, dans la variable 
history,  l'historique de l’entraînement  
2) Examiner le contenu de history 
3) Se servir de l’historique pour visualiser (graphique) et interpréter l’apprentissage

1) Entrainer le modèle avec 20 % des données d'entraînement utilisés pour la validation, 50 
passages sur les données et 16 observations traitées à la fois. Sauvegarder, dans la variable 
history,  l'historique de l’entraînement  

In [40]:
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=50,
    batch_size=16,
    verbose=0,
)

print("Entraînement terminé sur", len(history.history["loss"]), "epochs.")

Entraînement terminé sur 50 epochs.


2) Examiner le contenu de history 